In [ ]:
# Setup — all imports live here so the notebook executes top-to-bottom
# without reimporting mid-notebook. When running headlessly (CI, HPC),
# uncomment the ``matplotlib.use('Agg')`` line *before* the pyplot import
# so figures never need an interactive display.
import dataclasses
import logging
import os
import sys
from collections import Counter
from pathlib import Path

# import matplotlib
# matplotlib.use('Agg')  # enable for headless runs
import matplotlib.pyplot as plt
import mne
import numpy as np
import seaborn as sns
from matplotlib.colors import BoundaryNorm, ListedColormap
from mne.viz import plot_topomap
from scipy.sparse.csgraph import connected_components
from scipy.stats import pearsonr

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:  # noqa: B007
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

from independent_vector_analysis import iva_g  # noqa: E402
from sklearn.decomposition import PCA  # noqa: E402

from scripts.analysis_common import (  # noqa: E402
    FREQUENCY_BANDS,
    WAVELET_BAND_FREQ_RESOLUTION_HZ,
    analyzers_to_datasets,
    load_analyzers,
    wavelet_transform,
)
from src.analysis.wavelet_ica import zscore_by_time  # noqa: E402
from src.definitions.constants import ProjectPaths  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ConditionVariants,
    ExclusionCategories,
    MusicTypeVariants,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
%matplotlib inline

# IVA Decomposition of Wavelet Power — Time Components

## Naming Convention

Notebooks in this directory are named after the **independent
dimension** — the feature axis where the IVA components live. Here
the per-subject feature axis is **time** (each time sample is a
feature), so the components are 1-D **time profiles** per source.
The K-axis (the *dataset* axis IVA aligns scores across) is the
**subject** axis.

This is the **transposed companion** of
[`wavelet_iva_frequency_channel.ipynb`](wavelet_iva_frequency_channel.ipynb):
the two notebooks swap the roles of features and samples. There:
`(F × C)` were features, `T` were samples, so components lived in
`(F × C)` and scores in time. Here: `T` are features, `(F × C)` are
samples, so **components live in time** and **scores live in
`(F × C)`**.

We use **`scores`** for what comes out of `S = W @ X` (the
per-observation weights — the things IVA explicitly aligns across
datasets) and **`components`** for what comes out of
`W @ PCA_loadings` (the per-feature pattern of the demixing
matrix). Naming convention matches the ICA notebooks under
[`04-wavelet-ica-analysis/`](../04-wavelet-ica-analysis/).

## Scope

Run the **Independent Vector Analysis** algorithm on the 4-D wavelet
power tensor and stop. Downstream analyses live further down in this
notebook (mirroring the frequency × channel companion).

Unlike single-dataset ICA, **IVA decomposes K datasets jointly** while
preserving the dependency between corresponding scores across
datasets. Here each **subject is one dataset**, so IVA yields a set
of components that are *already aligned* across subjects — the kth
score in subject 1 corresponds to the kth score in subjects 2..K.

## Reshape

Each subject's wavelet tensor is flattened so that **time** is the
observation axis and **channels × frequencies** are the samples:

```
Input:   (n_subjects, n_channels, n_freqs, n_times)
Per-subject transpose + reshape:
         (n_times, n_channels, n_freqs)
         → (n_times,  n_channels × n_freqs)
           ── features ── ────── samples ──────
Stack:   (n_pca,  n_channels × n_freqs,  n_subjects)  — IVA layout (N, T, K)
```

IVA-G requires a **square** mixing matrix per dataset (`N == feature
dim`), so each subject's `(T, C*F)` matrix is first reduced with a
**per-subject PCA** to `N_PCA` components before stacking.

## Pipeline

1. Load wavelet power for the chosen condition / music type (cached).
2. Apply subject / channel / time subsets for fast iteration.
3. Z-score along time.
4. Per-subject reshape to `(T, C*F)` and per-subject PCA to `N_PCA`.
5. Stack into `(N_PCA, C*F, K)` and run **IVA-G**.
6. Recover per-subject scores in `(F, C)` sample space and per-subject
   component time profiles.

After the final cell the following variables are available:

| Variable | Shape | Description |
|----------|-------|-------------|
| `bb_data` | `(S, C, F, T)` | Raw 4-D wavelet power tensor |
| `bb_z` | `(S, C, F, T)` | Z-scored 4-D wavelet power tensor |
| `X_subjects` | `(S, T, C×F)` | Per-subject z-scored reshape (time as features) |
| `pcas` | list[`PCA`] | Per-subject fitted PCA objects |
| `X_pca` | `(N_PCA, C×F, S)` | PCA-reduced IVA input layout |
| `W` | `(N_PCA, N_PCA, S)` | Per-subject IVA demixing matrices |
| `cost` | `(n_iter,)` | IVA cost per iteration |
| `iva_scores` | `(S, N_PCA, F, C)` | Per-subject IVA scores reshaped to (freq, channel) |
| `iva_components` | `(S, N_PCA, T)` | Per-subject IVA component time profiles |

## Configuration

In [ ]:
# ── Experiment configuration ───────────────────────────────────
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPES = [MusicTypeVariants.CLASSICAL]  # single type for fast exploration
EXCLUSION_CATEGORIES = [ExclusionCategories.BAD_MUSIC, ExclusionCategories.ARTIFACTS]
PROCESS_AND_SAVE_DATA = False  # set True to re-process raw files

# ── Wavelet settings ─────────────────────────────────────────
REPRESENTATION = "power"
WAVELET_FREQ_MIN = min(lo for lo, _ in FREQUENCY_BANDS.values())
WAVELET_FREQ_MAX = max(hi for _, hi in FREQUENCY_BANDS.values())
WAVELET_N_FREQS = max(
    2,
    int(round((WAVELET_FREQ_MAX - WAVELET_FREQ_MIN) / WAVELET_BAND_FREQ_RESOLUTION_HZ))
    + 1,
)
FREQS = np.linspace(WAVELET_FREQ_MIN, WAVELET_FREQ_MAX, WAVELET_N_FREQS)

KEEP_FREQUENCY_DIM = True
RESHAPE_FREQUENCY_DIM = True  # -> (n_subjects, n_channels, n_freqs, n_times)

# ── Reuse / compute ──────────────────────────────────────────
REUSE_WAVELETS = True  # load from cache; set False to compute + save

# ── Subject subset ────────────────────────────────────────────
N_SUBJECTS_SUBSET: int | None = 5

# ── Channel and time subset ─────────────────────────────────────
N_CHANNELS_SUBSET: int | None = 32  # first N channels (from 195)
N_TIMES_SUBSET: int | None = 10000  # first N time samples

# ── IVA settings ──────────────────────────────────────────────
# This notebook is a functionality overview, not the production run —
# the values below are deliberately small so the whole pipeline finishes
# in seconds. Scale N_COMPONENTS_PCA and IVA_MAX_ITER back up when
# running a real analysis.
N_COMPONENTS_PCA = 10  # per-subject PCA dim before IVA (= N in IVA's (N, T, K))
IVA_OPT_APPROACH = "newton"  # 'gradient', 'newton', or 'quasi'
IVA_MAX_ITER = 64
IVA_W_DIFF_STOP = 1e-6
IVA_VERBOSE = True
IVA_RANDOM_STATE = 42  # seeds per-subject PCA + W_init

# ── Storage directory ─────────────────────────────────────────
# Mirror the 04 cache so wavelets are not recomputed when iterating on IVA.
WAVELET_DIR: Path = (
    ProjectPaths.NOTEBOOKS_DIR / "04-wavelet-ica-analysis" / "wavelet_cache"
)

# ── Downstream analysis settings ──────────────────────────────
N_COMPONENTS_SHOW = 5  # number of IVA components to visualise downstream
SAVE_PLOTS = True
# Plots directory follows the new naming convention: the independent
# dimension (where the components live) is `time`.
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "05-wavelet-iva-analysis"
    / "plots"
    / "broadband"
    / "iva_time"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Wavelet base directory : {WAVELET_DIR}")
print(f"Plots directory        : {PLOTS_DIR}")
print(
    f"Frequencies            : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({len(FREQS)} steps)"
)
print(f"Per-subject PCA dim    : {N_COMPONENTS_PCA}")
print(f"IVA optimisation       : {IVA_OPT_APPROACH}  (max_iter={IVA_MAX_ITER})")
print(f"Components to show     : {N_COMPONENTS_SHOW}")

## Data Loading

In [ ]:
analyzers = load_analyzers(
    MUSIC_TYPES,
    CONDITION,
    EXCLUSION_CATEGORIES,
    PROCESS_AND_SAVE_DATA,
    normalize_data=False,
)
datasets = analyzers_to_datasets(analyzers)

# Always limit to first N_SUBJECTS_SUBSET individuals
if N_SUBJECTS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:N_SUBJECTS_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_SUBJECTS_SUBSET} individuals.")

# Slice to channel subset
if N_CHANNELS_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :N_CHANNELS_SUBSET, :])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_CHANNELS_SUBSET} channels.")

# Slice to time subset
if N_TIMES_SUBSET is not None:
    datasets = {
        label: dataclasses.replace(ad, data=ad.data[:, :, :N_TIMES_SUBSET])
        for label, ad in datasets.items()
    }
    print(f"Using first {N_TIMES_SUBSET} time samples.")

print("Loaded datasets:", list(datasets.keys()))
for label, ad in datasets.items():
    print(
        f"  {label}: {ad.n_items} subjects, {ad.n_features} channels, "
        f"{ad.n_samples} samples"
    )

## Load or Compute Wavelet Transforms

Stored in `WAVELET_DIR/broadband/` (shared with the 04 ICA notebooks).

In [ ]:
broadband_datasets = wavelet_transform(
    datasets=datasets,
    freqs=FREQS,
    representation=REPRESENTATION,
    keep_frequency_dim=KEEP_FREQUENCY_DIM,
    reshape_frequency_dim=RESHAPE_FREQUENCY_DIM,
    wavelet_dir=WAVELET_DIR / "broadband",
    reuse_wavelets=REUSE_WAVELETS,
)
for label, ad in broadband_datasets.items():
    source = ad.metadata.get("loaded_from_wavelet_file", "computed_now")
    print(f"[broadband] {label}: shape={ad.data.shape}  source={source}")

## Dataset Selection

Change `LABEL` to switch between music types. The remaining cells use
`bb_data` (broadband wavelet power, 4-D) and derived quantities.

In [ ]:
LABEL = list(broadband_datasets.keys())[0]

bb_ad = broadband_datasets[LABEL]
bb_data = bb_ad.data  # (n_subjects, n_channels, n_freqs, n_times)
sfreq = bb_ad.sfreq

n_subjects, n_channels, n_freqs, n_times = bb_data.shape
time = np.arange(n_times) / sfreq

print(f"Dataset    : {LABEL}")
print(f"Shape      : {bb_data.shape}  (subjects × channels × freqs × times)")
print(f"Duration   : {n_times / sfreq:.1f} s  @  {sfreq} Hz")
print(f"Freq range : {FREQS[0]:.1f}–{FREQS[-1]:.1f} Hz ({n_freqs} steps)")

---
## Step 1 — Z-score and Per-Subject Reshape

**Z-scoring** normalises each `(subject, channel, frequency)` time
series to zero mean and unit variance so that IVA is not dominated
by high-power channels, subjects, or frequencies.

**Reshape** transposes each subject's tensor so that **time becomes
the feature axis** and `(C, F)` becomes the sample axis. The
`(S, C, F, T)` tensor is permuted to `(S, T, C, F)` and then
flattened to `(S, T, C×F)`. This is the natural transpose of the
frequency × channel companion notebook (where features = `C×F`,
samples = `T`).

| Quantity | Shape | Description |
|----------|-------|-------------|
| `bb_z` | `(S, C, F, T)` | Z-scored wavelet power |
| `X_subjects` | `(S, T, C×F)` | Per-subject z-scored reshape (time as features) |

In [ ]:
# Z-score along time: each (subject, channel, frequency) slice → mean=0, std=1
bb_z = zscore_by_time(bb_data)  # (S, C, F, T)

# Per-subject transpose + reshape: (S, C, F, T) → (S, T, C, F) → (S, T, C*F).
# Time becomes the feature axis; (C × F) becomes the sample axis.
n_samp = n_channels * n_freqs
X_subjects = bb_z.transpose(0, 3, 1, 2).reshape(
    n_subjects, n_times, n_samp
)  # (S, T, C*F)

print(f"Per-subject reshape    : {X_subjects.shape}  (subjects, time, C*F)")
print(f"  Features per subject : {n_times}  (= T)")
print(f"  Samples per subject  : {n_samp}  (C={n_channels} * F={n_freqs})")

---
## Step 2 — Per-Subject PCA Dimensionality Reduction

`iva_g` assumes a **square** mixing matrix per dataset, i.e. the
number of estimated scores equals the input dimensionality. Running
IVA directly on `T` features per subject would be prohibitively
expensive (and rank-deficient if `T > C*F`). Each subject's matrix
is reduced with its **own** PCA to `N_PCA` components, after which
the K subject matrices are stacked into IVA's `(N, T_samples, K)`
layout — where here `T_samples = C*F`.

Keeping per-subject PCAs (rather than a single shared PCA) preserves
subject-specific temporal subspaces, which is exactly what IVA
exploits to align cross-subject scores.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `pcas[k]` | — | Fitted PCA object for subject `k` |
| `pca_evr` | `(S, N_PCA)` | Explained-variance ratio per subject |
| `X_pca` | `(N_PCA, C*F, S)` | IVA input layout (N, T_samples, K) |

In [ ]:
# Per-subject PCA. PCA expects (n_samples, n_features); each subject's matrix
# is (T, C*F) with T = features → transpose to (C*F, T), fit, transform back
# to (C*F, N_PCA), transpose to (N_PCA, C*F), then stack along axis 2 as IVA
# expects.
pcas: list[PCA] = []
pca_scores_per_subject = np.zeros((n_subjects, N_COMPONENTS_PCA, n_samp))
pca_evr = np.zeros((n_subjects, N_COMPONENTS_PCA))

for k in range(n_subjects):
    subj_matrix = X_subjects[k].T  # (C*F, T) — samples × features for sklearn
    pca = PCA(n_components=N_COMPONENTS_PCA, random_state=IVA_RANDOM_STATE)
    scores = pca.fit_transform(subj_matrix)  # (C*F, N_PCA)
    pcas.append(pca)
    pca_scores_per_subject[k] = scores.T  # (N_PCA, C*F)
    pca_evr[k] = pca.explained_variance_ratio_
    print(
        f"  S{k + 1}: explained variance = {pca_evr[k].sum() * 100:5.1f}% "
        f"({N_COMPONENTS_PCA} comps)"
    )

# IVA expects (N, T_samples, K)
X_pca = np.ascontiguousarray(pca_scores_per_subject.transpose(1, 2, 0))

print(f"\nIVA input shape        : {X_pca.shape}  (N_PCA, C*F, K=subjects)")

# Quick visual of how much variance each subject's PCA retains.
fig, ax = plt.subplots(figsize=(8, 4))
for k in range(n_subjects):
    ax.plot(
        np.arange(1, N_COMPONENTS_PCA + 1),
        np.cumsum(pca_evr[k]),
        marker="o",
        markersize=3,
        label=f"S{k + 1}",
    )
ax.set_xlabel("Number of PCA components")
ax.set_ylabel("Cumulative variance explained")
ax.set_title(f"Per-Subject PCA — {LABEL}")
ax.legend(fontsize=8, ncol=2)
ax.axhline(0.9, ls="--", lw=0.6, color="gray")
fig.tight_layout()
plt.show()
plt.close("all")

---
## Step 3 — Run IVA-G

`iva_g` returns a demixing matrix `W` of shape `(N, N, K)`. The
scores for subject `k` are then

```
S_pca[k]      = W[:, :, k] @ X_pca[:, :, k]           # (N_PCA, C*F)
S_full[k]     = pcas[k].components_.T @ S_pca[k]       # (T, C*F)
components[k] = W[:, :, k] @ pcas[k].components_       # (N_PCA, T)
```

Because IVA's permutation ambiguity is **shared across datasets**, the
kth score in `S_full[0]` corresponds to the kth score in
`S_full[1..K-1]` — there is no need to align scores across subjects
post-hoc.

| Returned by `iva_g` | Shape | Description |
|---------------------|-------|-------------|
| `W` | `(N_PCA, N_PCA, S)` | Per-subject demixing matrix |
| `cost` | `(n_iter,)` | IVA cost per iteration |
| `Sigma_N` | `(S, S, N_PCA)` | Per-score-component covariance across subjects |
| `isi` | `float` | Joint inter-symbol-interference (only when ground-truth `A` is given) |

In [ ]:
# Deterministic W initialisation: random matrix per subject seeded by IVA_RANDOM_STATE.
rng = np.random.default_rng(IVA_RANDOM_STATE)
W_init = rng.standard_normal((N_COMPONENTS_PCA, N_COMPONENTS_PCA, n_subjects))

W, cost, Sigma_N, isi = iva_g(
    X_pca,
    opt_approach=IVA_OPT_APPROACH,
    whiten=True,
    verbose=IVA_VERBOSE,
    W_init=W_init,
    max_iter=IVA_MAX_ITER,
    W_diff_stop=IVA_W_DIFF_STOP,
)

print(f"\nW shape           : {W.shape}  (N_PCA, N_PCA, K=subjects)")
print(f"Sigma_N shape     : {Sigma_N.shape}  (K, K, N_PCA)")
print(f"Iterations        : {len(cost)}")
print(f"Final cost        : {cost[-1]:.6f}")

# Cost-curve sanity check.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(cost, marker="o", markersize=3, color="steelblue")
ax.set_xlabel("Iteration")
ax.set_ylabel("IVA cost")
ax.set_title(f"IVA-G Convergence — {LABEL}")
fig.tight_layout()
plt.show()
plt.close("all")

---
## Step 4 — Recover Scores and Component Patterns

Apply the per-subject demixing matrix to the PCA-reduced data to
obtain IVA scores (which live in the `(C*F)` sample space and are
reshaped to `(F, C)`), then combine `W_k` with the PCA loadings to
express each component as a **time profile** over the original `T`
feature space.

The kth row of `iva_components[k]` is the kth source's time profile
in subject `k`'s original time axis. Because IVA resolves the
permutation jointly across datasets, the kth row is **already
aligned** across subjects.

In [ ]:
iva_components = np.zeros((n_subjects, N_COMPONENTS_PCA, n_times))
iva_scores = np.zeros((n_subjects, N_COMPONENTS_PCA, n_freqs, n_channels))

for k in range(n_subjects):
    W_k = W[:, :, k]  # (N_PCA, N_PCA)
    X_pca_k = X_pca[:, :, k]  # (N_PCA, C*F)

    # Scores in the (C*F) sample subspace, then reshaped to (F, C).
    scores_flat = W_k @ X_pca_k  # (N_PCA, C*F)

    # X_subjects was built by (S, C, F, T) → (S, T, C, F) → (S, T, C*F),
    # so the flattened (C*F) axis iterates channels (slow) then frequencies (fast).
    # Reshape back to (N_PCA, C, F) and transpose to (N_PCA, F, C) to mirror
    # the 04 ICA notebook's components_2d layout.
    iva_scores[k] = scores_flat.reshape(
        N_COMPONENTS_PCA, n_channels, n_freqs
    ).transpose(0, 2, 1)

    # Components in the full T feature space: rows of W projected through PCA.
    iva_components[k] = W_k @ pcas[k].components_  # (N_PCA, T)

print(f"IVA components (time)    : {iva_components.shape}   (S, N_PCA, T)")
print(f"IVA scores (F, C)        : {iva_scores.shape}    (S, N_PCA, F, C)")

---
## Result Summary

All variables required for downstream analyses are now in memory:

| Variable | Shape | Description |
|----------|-------|-------------|
| `bb_data` | `(S, C, F, T)` | Raw 4-D wavelet power tensor |
| `bb_z` | `(S, C, F, T)` | Z-scored wavelet power tensor |
| `X_subjects` | `(S, T, C×F)` | Per-subject z-scored reshape (time as features) |
| `pcas` | list[`PCA`] | Per-subject fitted PCA objects |
| `pca_evr` | `(S, N_PCA)` | Per-subject PCA explained-variance ratio |
| `X_pca` | `(N_PCA, C×F, S)` | IVA input (PCA-reduced) |
| `W` | `(N_PCA, N_PCA, S)` | Per-subject IVA demixing |
| `Sigma_N` | `(S, S, N_PCA)` | Per-SCV covariance across subjects |
| `cost` | `(n_iter,)` | IVA convergence trace |
| `iva_components` | `(S, N_PCA, T)` | Component time profiles |
| `iva_scores` | `(S, N_PCA, F, C)` | Scores reshaped to (freq, channel) |

The kth score/component is **aligned across subjects** by
construction — no post-hoc matching is required. IVA's explicit
alignment guarantee is on **scores** (the `(F × C)` sample space
here); components inherit the same kth-row identity by association.

# Downstream Analyses

The cells below mirror the ISC-style plots from the companion ICA
notebook
[`04-wavelet-ica-analysis/wavelet_ica_frequency_channel.ipynb`](../04-wavelet-ica-analysis/wavelet_ica_frequency_channel.ipynb)
and from the frequency × channel sibling
[`wavelet_iva_frequency_channel.ipynb`](wavelet_iva_frequency_channel.ipynb).
Because **time is the independent dimension** here, the roles of
*scores* and *components* are **swapped** relative to the sibling:

| Variable | Shape | Independent axis | Per-subject vector for ISC |
|----------|-------|------------------|----------------------------|
| `iva_components` | `(S, N_PCA, T)` | **time** (`T` samples) | `iva_components[s, k, :]` — length `T` |
| `iva_scores` | `(S, N_PCA, F, C)` | **frequency × channel** (`F × C` features) | `iva_scores[s, k].ravel()` — length `F·C` |

So in everything that follows:

- **components ⇔ time dimension** (the kth component's temporal
  profile across subjects). This is the **independent dimension**
  of this notebook — the axis where the components live.
- **scores ⇔ frequency × channel dimension** (the kth source's
  per-`(f, c)` activation across subjects). This is the IVA sample
  axis here, so IVA explicitly aligns the kth score across subjects
  in this space.

Each downstream plot shows the first `N_COMPONENTS_SHOW` components
(currently `5`), and all plots save under `PLOTS_DIR`
(`plots/broadband/iva_time/`).

In [ ]:
# Pearson-r thresholds used for the cluster strip below each ISC matrix.
ISC_CLUSTER_THRESHOLDS = (0.3, 0.5, 0.7)

# Discrete colormap: light gray for singletons (0) + tab10 for groups (1..10).
_GROUP_PALETTE = list(plt.colormaps["tab10"].colors)
_CLUSTER_CMAP = ListedColormap(["#dddddd"] + _GROUP_PALETTE)
_CLUSTER_NORM = BoundaryNorm(
    np.arange(-0.5, len(_GROUP_PALETTE) + 1.5, 1.0), _CLUSTER_CMAP.N
)


def _cluster_grid(
    corr_mat: np.ndarray, thresholds=ISC_CLUSTER_THRESHOLDS
) -> np.ndarray:
    """(T, S) grid of within-row cluster IDs. Singletons → 0, groups → 1, 2, ...."""
    grid = np.zeros((len(thresholds), n_subjects), dtype=int)
    for t_idx, thr in enumerate(thresholds):
        adj = (corr_mat >= thr) & ~np.eye(n_subjects, dtype=bool)
        _, labels = connected_components(adj, directed=False)
        counts = Counter(labels.tolist())
        next_group = 1
        group_map: dict[int, int] = {}
        for s in range(n_subjects):
            lab = int(labels[s])
            if counts[lab] == 1:
                grid[t_idx, s] = 0
            else:
                if lab not in group_map:
                    group_map[lab] = next_group
                    next_group += 1
                grid[t_idx, s] = group_map[lab]
    return grid


def _plot_isc_grid(corr_per_comp, fig_title, file_name):
    """Plot ISC correlation matrices + cluster strips for the first N components."""
    n_show = N_COMPONENTS_SHOW
    fig, axes = plt.subplots(
        2,
        n_show,
        figsize=(2.8 * n_show, 5.5),
        gridspec_kw={"height_ratios": [3, 1.2]},
        constrained_layout=True,
    )
    if n_show == 1:
        axes = axes.reshape(2, 1)

    im_corr = None
    for i in range(n_show):
        corr_mat = corr_per_comp[i]
        ax_top = axes[0, i]
        im_corr = ax_top.imshow(corr_mat, vmin=-1, vmax=1, cmap="RdBu_r")
        ax_top.set_xticks(range(n_subjects))
        ax_top.set_yticks(range(n_subjects))
        ax_top.set_xticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
        ax_top.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
        ax_top.set_title(f"IC {i + 1}", fontsize=10)

        ax_bot = axes[1, i]
        grid = _cluster_grid(corr_mat)
        ax_bot.imshow(grid, cmap=_CLUSTER_CMAP, norm=_CLUSTER_NORM, aspect="auto")
        for ti in range(grid.shape[0]):
            for sj in range(grid.shape[1]):
                val = int(grid[ti, sj])
                if val > 0:
                    ax_bot.text(
                        sj,
                        ti,
                        str(val),
                        ha="center",
                        va="center",
                        fontsize=7,
                        color="white",
                        weight="bold",
                    )
        ax_bot.set_xticks(range(n_subjects))
        ax_bot.set_xticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=7)
        ax_bot.set_yticks(range(len(ISC_CLUSTER_THRESHOLDS)))
        ax_bot.set_yticklabels(
            [f"r≥{thr}" for thr in ISC_CLUSTER_THRESHOLDS], fontsize=8
        )
        if i == 0:
            ax_bot.set_ylabel("Threshold")

    fig.suptitle(fig_title, fontsize=12)
    fig.colorbar(im_corr, ax=axes[0, -1], label="Pearson r", shrink=0.8)
    if SAVE_PLOTS:
        fig.savefig(PLOTS_DIR / file_name, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close("all")

---
## Analysis (a) — PCA Scree Plot

IVA uses **per-subject PCA** (cell `iva-13-pca`) to reduce the
`(C × F)` feature space before joint decomposition, so the scree plot
summarises explained variance across **all subjects**:

- **Left bar plot**: across-subject mean of
  `explained_variance_ratio_` per component, with std as error bars.
- **Right line plot**: per-subject cumulative variance (gray, thin)
  + across-subject mean cumulative (coral) with a 90 % reference.

The red dashed line marks the cutoff at `N_COMPONENTS_SHOW`, the
number of components used in the downstream analyses below.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `pca_evr_mean` | `(N_PCA,)` | Mean explained variance ratio across subjects |
| `pca_evr_std` | `(N_PCA,)` | Std of explained variance ratio across subjects |
| `pca_evr_cum_mean` | `(N_PCA,)` | Mean cumulative explained variance |

In [ ]:
# Across-subject summary of per-subject PCA explained-variance ratios.
pca_evr_mean = pca_evr.mean(axis=0)  # (N_PCA,)
pca_evr_std = pca_evr.std(axis=0)  # (N_PCA,)
pca_evr_cum_mean = np.cumsum(pca_evr_mean)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

xs = np.arange(1, N_COMPONENTS_PCA + 1)
axes[0].bar(xs, pca_evr_mean, yerr=pca_evr_std, color="steelblue", capsize=2)
axes[0].axvline(
    N_COMPONENTS_SHOW + 0.5,
    ls="--",
    color="firebrick",
    label=f"first {N_COMPONENTS_SHOW}",
)
axes[0].set_xlabel("PCA component")
axes[0].set_ylabel("Explained variance ratio")
axes[0].set_title(f"PCA Scree (mean ± std across subjects) — {LABEL}")
axes[0].legend()

for k in range(n_subjects):
    axes[1].plot(xs, np.cumsum(pca_evr[k]), lw=0.6, alpha=0.5, color="gray")
axes[1].plot(xs, pca_evr_cum_mean, "o-", color="coral", label="mean cumulative")
axes[1].axhline(0.9, ls="--", color="gray", label="90%")
axes[1].axvline(
    N_COMPONENTS_SHOW + 0.5,
    ls="--",
    color="firebrick",
    label=f"first {N_COMPONENTS_SHOW}",
)
axes[1].set_xlabel("Number of components")
axes[1].set_ylabel("Cumulative variance explained")
axes[1].set_title(f"Cumulative Variance — {LABEL}")
axes[1].legend()

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "pca_scree.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

print(
    f"First {N_COMPONENTS_SHOW} components explain "
    f"{pca_evr_cum_mean[N_COMPONENTS_SHOW - 1] * 100:.1f}% of variance on average."
)

---
## Analysis (b) — Intersubject Correlation of Component Timecourses (Time Dimension)

**Components ⇔ time.** For each IVA component the per-subject
vector is the length-`T` row `iva_components[s, k, :]` — the kth
source's time profile for subject `s` (this is the **independent
dimension** of this notebook). We build a subject × subject Pearson
correlation matrix per component over this time axis. High
off-diagonal correlations reflect a temporally consistent activation
pattern.

A **cluster strip** beneath each correlation matrix colors subjects
by connected-component membership at each ISC threshold (`r ≥ 0.3,
0.5, 0.7`). Singletons stay gray; numeric group IDs disambiguate
clusters within a row. This mirrors the cluster strip used in
`04-wavelet-ica-analysis/wavelet_ica_frequency_channel.ipynb`.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `component_corr_per_comp` | `(N_SHOW, S, S)` | Per-IC subject × subject component correlation along the time axis |

In [ ]:
# Subject × subject correlations of component timecourses, per component.
component_corr_per_comp = np.stack(
    [np.corrcoef(iva_components[:, k, :]) for k in range(N_COMPONENTS_SHOW)]
)  # (N_SHOW, S, S)

_plot_isc_grid(
    component_corr_per_comp,
    fig_title=f"Intersubject Correlation of IVA Component Timecourses — {LABEL}",
    file_name="iva_component_isc_time.png",
)

---
## Analysis (c) — Intersubject Correlation of Score Patterns (Frequency × Channel Dimension)

**Scores ⇔ frequency × channel.** Same construction as (b), but the
per-subject vector for component `k` is now the flattened
`(F × C)` pattern `iva_scores[s, k].ravel()` — the kth source's
spatio-spectral fingerprint for subject `s`. Because IVA explicitly
aligns the kth score across subjects in this `(F × C)` sample
space, high off-diagonal correlations are expected here; (b) above
is the complementary view in the time feature axis where no such
alignment is enforced.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `pattern_corr_per_comp` | `(N_SHOW, S, S)` | Per-IC subject × subject score-pattern correlation along the (F × C) axis |

In [ ]:
# Subject × subject correlations of (F, C) score patterns, per IC.
# Each subject's pattern is iva_scores[s, k] of shape (F, C), flattened to a
# length-(F*C) vector so the correlation is taken over the IVA sample axis.
pattern_corr_per_comp = np.zeros((N_COMPONENTS_SHOW, n_subjects, n_subjects))
for k in range(N_COMPONENTS_SHOW):
    flat_patterns = iva_scores[:, k, :, :].reshape(n_subjects, -1)  # (S, F*C)
    pattern_corr_per_comp[k] = np.corrcoef(flat_patterns)

_plot_isc_grid(
    pattern_corr_per_comp,
    fig_title=(
        f"Intersubject Correlation of IVA Score (F × C) Patterns — {LABEL}"
    ),
    file_name="iva_score_isc_freq_channel.png",
)

---
## Analysis (d) — Mean LOO-ISC per Component (Time Dimension)

Whole-recording leave-one-out inter-subject correlation per IVA
component over the **time** axis — i.e. computed from the
per-subject component timecourses `iva_components[s, k, :]`. For
each component `k`:

```
for each subject s:
    others_mean = mean over s' ≠ s of iva_components[s', k, :]
    r[k, s]     = pearson(iva_components[s, k, :], others_mean)
```

The bar height is the across-subject **mean** of `r[k, s]`; the
error bar is the across-subject **std**. Bars are red where the
mean is negative. Pair this with (e) below for the matching summary
along the frequency × channel axis (where IVA explicitly aligns).

| Quantity | Shape | Description |
|----------|-------|-------------|
| `loo_isc_time_per_subject` | `(N_SHOW, S)` | Whole-recording LOO-ISC per IC and subject (time axis) |
| `loo_isc_time_mean` | `(N_SHOW,)` | Across-subject mean LOO-ISC per IC (time axis) |
| `loo_isc_time_std` | `(N_SHOW,)` | Across-subject std LOO-ISC per IC (time axis) |

In [ ]:
# Time-dimension LOO-ISC: per-subject vector = component timecourse
# iva_components[s, k, :].
loo_isc_time_per_subject = np.zeros((N_COMPONENTS_SHOW, n_subjects))
for k in range(N_COMPONENTS_SHOW):
    subj_vectors = iva_components[:, k, :]  # (S, T)
    for s in range(n_subjects):
        others_mean = np.delete(subj_vectors, s, axis=0).mean(axis=0)
        loo_isc_time_per_subject[k, s] = float(
            pearsonr(subj_vectors[s], others_mean)[0]
        )

loo_isc_time_mean = loo_isc_time_per_subject.mean(axis=1)
loo_isc_time_std = loo_isc_time_per_subject.std(axis=1)

bar_colors = ["firebrick" if m < 0 else "steelblue" for m in loo_isc_time_mean]

fig, ax = plt.subplots(figsize=(max(8, 0.9 * N_COMPONENTS_SHOW), 4.5))
xs = np.arange(N_COMPONENTS_SHOW)
ax.bar(xs, loo_isc_time_mean, yerr=loo_isc_time_std, color=bar_colors, capsize=4)
ax.axhline(0.0, ls="--", lw=0.6, color="gray")
ax.set_xticks(xs)
ax.set_xticklabels([f"IC {k + 1}" for k in range(N_COMPONENTS_SHOW)])
ax.set_xlabel("Component")
ax.set_ylabel("Mean LOO-ISC across subjects")
ax.set_ylim(-1.05, 1.05)
ax.set_title(
    f"Per-IC Mean LOO-ISC — Time Dimension (component timecourses) — {LABEL}"
)

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "iva_loo_isc_bar_time.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (e) — Mean LOO-ISC per Component (Frequency × Channel Dimension)

Companion to (d) along the IVA-aligned sample axis of this
notebook. Whole-recording LOO-ISC per IVA component using the
per-subject score pattern flattened over `(F × C)`. For each
component `k`:

```
for each subject s:
    pat[s']     = iva_scores[s', k, :, :].ravel()  # length F*C
    others_mean = mean over s' ≠ s of pat[s']
    r[k, s]     = pearson(pat[s], others_mean)
```

Same bar / errorbar / sign-coloring conventions as (d). High bars
here are expected because IVA explicitly aligns the kth score in
the `(F × C)` sample space; (d) shows the analogous quantity in
the time feature axis where no such alignment is enforced.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `loo_isc_fc_per_subject` | `(N_SHOW, S)` | Whole-recording LOO-ISC per IC and subject (F × C axis) |
| `loo_isc_fc_mean` | `(N_SHOW,)` | Across-subject mean LOO-ISC per IC (F × C axis) |
| `loo_isc_fc_std` | `(N_SHOW,)` | Across-subject std LOO-ISC per IC (F × C axis) |

In [ ]:
# (F × C)-dimension LOO-ISC: per-subject vector = flattened score pattern
# iva_scores[s, k].ravel() (length F*C).
loo_isc_fc_per_subject = np.zeros((N_COMPONENTS_SHOW, n_subjects))
for k in range(N_COMPONENTS_SHOW):
    subj_vectors = iva_scores[:, k, :, :].reshape(n_subjects, -1)  # (S, F*C)
    for s in range(n_subjects):
        others_mean = np.delete(subj_vectors, s, axis=0).mean(axis=0)
        loo_isc_fc_per_subject[k, s] = float(
            pearsonr(subj_vectors[s], others_mean)[0]
        )

loo_isc_fc_mean = loo_isc_fc_per_subject.mean(axis=1)
loo_isc_fc_std = loo_isc_fc_per_subject.std(axis=1)

bar_colors = ["firebrick" if m < 0 else "steelblue" for m in loo_isc_fc_mean]

fig, ax = plt.subplots(figsize=(max(8, 0.9 * N_COMPONENTS_SHOW), 4.5))
xs = np.arange(N_COMPONENTS_SHOW)
ax.bar(xs, loo_isc_fc_mean, yerr=loo_isc_fc_std, color=bar_colors, capsize=4)
ax.axhline(0.0, ls="--", lw=0.6, color="gray")
ax.set_xticks(xs)
ax.set_xticklabels([f"IC {k + 1}" for k in range(N_COMPONENTS_SHOW)])
ax.set_xlabel("Component")
ax.set_ylabel("Mean LOO-ISC across subjects")
ax.set_ylim(-1.05, 1.05)
ax.set_title(
    f"Per-IC Mean LOO-ISC — Frequency × Channel Dimension (score patterns) — {LABEL}"
)

fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "iva_loo_isc_bar_freq_channel.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

---
## Analysis (f) — Topomap of Mean and Variance Across Subjects

Computed from **`iva_scores`** (the components have no channel
axis, so this is the only way to project the kth source onto the
scalp). For every component `k` each subject has their own
`(F × C)` score pattern; collapsing the frequency axis yields a
per-subject **channel loading** vector:

```
chan_loading[s, k, c] = mean over f of iva_scores[s, k, f, c]
```

We then summarise the across-subject distribution per channel with
the **mean** and the **variance**, giving two topomaps per
component: the mean spatial fingerprint (`RdBu_r`, symmetric) and
the across-subject dispersion (`viridis`, non-negative).

| Quantity | Shape | Description |
|----------|-------|-------------|
| `chan_loading` | `(S, N_PCA, C)` | Per-subject channel loading per IC |
| `chan_mean` | `(N_PCA, C)` | Mean channel loading across subjects |
| `chan_var` | `(N_PCA, C)` | Variance of channel loading across subjects |

In [ ]:
# Per-subject channel loading (collapse frequency axis): (S, N_PCA, C)
chan_loading = iva_scores.mean(axis=2)
chan_mean = chan_loading.mean(axis=0)  # (N_PCA, C)
chan_var = chan_loading.var(axis=0)  # (N_PCA, C)

# MNE info for the topomap, restricted to the channel subset.
info = analyzers[LABEL].info
info = mne.pick_info(info, mne.pick_types(info, eeg=True))
if n_channels < len(info.ch_names):
    info = mne.pick_info(info, list(range(n_channels)))

n_show = N_COMPONENTS_SHOW
fig, axes = plt.subplots(2, n_show, figsize=(3.0 * n_show, 7.0))
if n_show == 1:
    axes = axes.reshape(2, 1)

for i in range(n_show):
    # Mean topomap (signed)
    vlim_m = float(np.percentile(np.abs(chan_mean[i]), 99))
    if vlim_m == 0.0:
        vlim_m = 1e-12
    im_m, _ = plot_topomap(
        chan_mean[i],
        info,
        axes=axes[0, i],
        show=False,
        cmap="RdBu_r",
        vlim=(-vlim_m, vlim_m),
    )
    axes[0, i].set_title(f"IC {i + 1}", fontsize=10)
    fig.colorbar(im_m, ax=axes[0, i], fraction=0.046, pad=0.04)

    # Variance topomap (non-negative)
    vlim_v = float(np.percentile(chan_var[i], 99))
    if vlim_v == 0.0:
        vlim_v = 1e-12
    im_v, _ = plot_topomap(
        chan_var[i],
        info,
        axes=axes[1, i],
        show=False,
        cmap="viridis",
        vlim=(0.0, vlim_v),
    )
    fig.colorbar(im_v, ax=axes[1, i], fraction=0.046, pad=0.04)

fig.text(
    0.01,
    0.75,
    "Mean across subjects",
    rotation=90,
    va="center",
    fontsize=11,
    fontweight="bold",
)
fig.text(
    0.01,
    0.25,
    "Variance across subjects",
    rotation=90,
    va="center",
    fontsize=11,
    fontweight="bold",
)
fig.suptitle(
    f"Mean and Variance Topomaps Across Subjects — {LABEL}",
    fontsize=13,
)
fig.tight_layout(rect=(0.03, 0, 1, 0.97))
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "iva_topomap_mean_var.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (g) — Time × Frequency Map per Component (Mean Across Subjects)

Combines both ingredients we have: the **frequency profile** is the
subject-averaged, channel-averaged loading from `iva_scores`, and
the **time profile** is the subject-averaged component timecourse
from `iva_components`. Their outer product gives a rank-1
approximation of the component's `(F, T)` expression that is
grounded entirely in the IVA solution — channels are averaged out
of the scores, and subjects are averaged out of both axes.

```
freq_profile[k] = iva_scores.mean(axis=0)[k].mean(axis=1)   # (F,) channel-avg
time_profile[k] = iva_components.mean(axis=0)[k]            # (T,) subject-avg
tf_map[k]       = outer(freq_profile[k], time_profile[k])   # (F, T)
```

| Quantity | Shape | Description |
|----------|-------|-------------|
| `freq_profiles` | `(F, N_PCA)` | Channel-averaged loading per frequency, per IC (mean across subjects) |
| `time_profiles` | `(N_PCA, T)` | Subject-averaged component timecourse per IC |
| `ft_maps` | `(N_PCA, F, T)` | Outer-product frequency × time map per component |

In [ ]:
# Subject-averaged spatial-spectral pattern: (N_PCA, F, C)
mean_scores = iva_scores.mean(axis=0)
# Channel-averaged frequency profile, transposed to (F, N_PCA) for einsum
freq_profiles = mean_scores.mean(axis=2).T  # (F, N_PCA)
# Subject-averaged component timecourse: (N_PCA, T)
time_profiles = iva_components.mean(axis=0)

# Outer product per component → (N_PCA, F, T)
ft_maps = np.einsum("fk,kt->kft", freq_profiles, time_profiles)

n_show = N_COMPONENTS_SHOW
fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.6 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    data_i = ft_maps[i]  # (F, T)
    vlim_i = np.percentile(np.abs(data_i), 99)
    mesh = ax.pcolormesh(
        time,
        FREQS,
        data_i,
        cmap="RdBu_r",
        vmin=-vlim_i,
        vmax=vlim_i,
        shading="auto",
    )
    ax.set_ylabel("Freq (Hz)")
    ax.set_title(f"IC {i + 1} — Time × Frequency (outer product)", fontsize=10)
    fig.colorbar(mesh, ax=ax, pad=0.01, fraction=0.025)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"Per-IC Time × Frequency Maps (mean across subjects) — {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "iva_time_frequency.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close("all")

---
## Analysis (h) — Component Timecourses: Mean and Variance Across Subjects

Per-component **temporal resolution**: for each IVA component the
per-subject timecourse `iva_components[s, k, :]` is collapsed
across the subject axis at every time point:

```
mean_temporal[k, t] = mean over s of iva_components[s, k, t]   # (K, T)
var_temporal[k, t]  = var  over s of iva_components[s, k, t]   # (K, T)
std_temporal[k, t]  = sqrt(var_temporal[k, t])                 # (K, T)
```

The **mean** is plotted as a line and the **variance** as an
error-shading band (`mean ± √variance`) around it. Narrow bands
mark time points where subjects agree on the kth component's
expression; wide bands mark idiosyncratic intervals.

Note: unlike the frequency × channel sibling notebook (where the
score timecourses inherited IVA's explicit alignment), the
component timecourses here come from `W @ PCA_loadings` and carry
no direct IVA alignment guarantee — wide bands are therefore more
informative.

| Quantity | Shape | Description |
|----------|-------|-------------|
| `mean_temporal` | `(N_PCA, T)` | Mean component across subjects |
| `var_temporal` | `(N_PCA, T)` | Variance of component across subjects |
| `std_temporal` | `(N_PCA, T)` | Std (band width) |

In [ ]:
# Across-subject summaries of each IVA component's timecourse.
mean_temporal = iva_components.mean(axis=0)  # (N_PCA, T)
var_temporal = iva_components.var(axis=0)  # (N_PCA, T)
std_temporal = np.sqrt(var_temporal)  # used as error-band width

n_show = N_COMPONENTS_SHOW
fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.5 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.plot(time, mean_temporal[i], lw=0.9, color="darkorange", label="mean")
    ax.fill_between(
        time,
        mean_temporal[i] - std_temporal[i],
        mean_temporal[i] + std_temporal[i],
        alpha=0.25,
        color="darkorange",
        label="± √variance",
    )
    ax.set_ylabel(f"IC {i + 1}")
    ax.set_title(
        f"Component {i + 1} — Mean & Variance Across Subjects", fontsize=10
    )
    if i == 0:
        ax.legend(loc="upper right", fontsize=8)

axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"Per-IC Component Timecourse — Mean and Variance Across Subjects — {LABEL}",
    fontsize=13,
    y=1.01,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "iva_mean_variance_over_time.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

---
## Analysis (i) — Pair-Space Heatmaps per Component

For every component `k` we show four `(X × Y)` heatmaps — **first
name in the title is the x-axis, second is the y-axis**, following
the 04 ICA notebook convention. The axis priority is:

1. **time** is on x whenever it's in the pair (matches 04 ICA cell 31, "Time × Subject");
2. if there's no time, **subject** is on x;
3. if there's neither time nor subject, **frequency** is on x (matches 04 ICA cell 33, "Frequency × Channel").

The third dimension is collapsed by averaging. Sources are swapped
relative to the frequency × channel sibling: the time view comes
from `iva_components` (since components live in time here) and the
spatial views from `iva_scores`.

| Variant | Source | Collapsed axis | x | y |
|---------|--------|----------------|----|----|
| **Time × Subject** | `iva_components[:, k, :]` | — (components have no F/C) | time | subject |
| **Subject × Frequency** | `iva_scores[:, k, :, :].mean(axis=-1)` | channel | subject | frequency |
| **Subject × Channel** | `iva_scores[:, k, :, :].mean(axis=-2)` | frequency | subject | channel |
| **Frequency × Channel** | `iva_scores[:, k, :, :].mean(axis=0)` | subject | frequency | channel |

A diverging colormap (`RdBu_r`, symmetric around zero) preserves
the sign in every panel. Color limits are **shared across
components** for the three subject-involving variants (so
components are visually comparable on that variant — matches 04
ICA cell 31), and **per-component** for the Frequency × Channel
variant so weaker components are not flattened by stronger ones
(matches 04 ICA cell 33).

| Quantity | Shape | Description |
|----------|-------|-------------|
| `time_subj` | `(N_SHOW, S, T)` | Per-IC matrices for Time × Subject (rows = subject = y, cols = time = x) |
| `subj_freq` | `(N_SHOW, F, S)` | Per-IC matrices for Subject × Frequency (rows = frequency = y, cols = subject = x) |
| `subj_chan` | `(N_SHOW, C, S)` | Per-IC matrices for Subject × Channel (rows = channel = y, cols = subject = x) |
| `freq_chan` | `(N_SHOW, F, C)` | Per-IC subject-averaged `(F, C)` patterns; transposed to `(C, F)` at plot time |

In [ ]:
n_show = N_COMPONENTS_SHOW
channels = np.arange(n_channels)
subjects_idx = np.arange(n_subjects)
subject_labels = [f"S{s + 1}" for s in subjects_idx]


def _safe_vlim(arr: np.ndarray) -> float:
    """99th percentile of |arr|, never zero (so vmin/vmax stay valid)."""
    return max(float(np.percentile(np.abs(arr), 99)), 1e-12)


# ── (1) Time × Subject — x=time, y=subject. Matches 04 ICA cell 31. ──────
# Source: iva_components (components have no F/C axis to collapse).
time_subj = iva_components.transpose(1, 0, 2)[:n_show]  # (K, S, T)
_vlim = _safe_vlim(time_subj)

fig, axes = plt.subplots(n_show, 1, figsize=(14, 2.6 * n_show), sharex=True)
if n_show == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    mesh = ax.pcolormesh(
        time,
        subjects_idx,
        time_subj[i],
        cmap="RdBu_r",
        vmin=-_vlim,
        vmax=_vlim,
        shading="auto",
    )
    ax.set_yticks(subjects_idx)
    ax.set_yticklabels(subject_labels, fontsize=8)
    ax.set_ylabel("Subject")
    ax.set_title(f"IC {i + 1} — Time × Subject", fontsize=10)
    fig.colorbar(mesh, ax=ax, pad=0.01, fraction=0.025, label="component")
axes[-1].set_xlabel("Time (s)")
fig.suptitle(
    f"Time × Subject per IC (from components) — {LABEL}", fontsize=13, y=1.01
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "iva_pairmap_time_subject.png", dpi=150, bbox_inches="tight"
    )
plt.show()
plt.close("all")


# ── (2) Subject × Frequency — x=subject, y=frequency. Channel-avg scores. ─
# iva_scores.mean(axis=-1) → (S, K, F); transpose to (K, F, S) for plot.
subj_freq = iva_scores.mean(axis=-1).transpose(1, 2, 0)[:n_show]  # (K, F, S)
_vlim = _safe_vlim(subj_freq)

fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 5.0), sharey=True)
if n_show == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    mesh = ax.pcolormesh(
        subjects_idx,
        FREQS,
        subj_freq[i],
        cmap="RdBu_r",
        vmin=-_vlim,
        vmax=_vlim,
        shading="auto",
    )
    ax.set_xticks(subjects_idx)
    ax.set_xticklabels(subject_labels, fontsize=8)
    ax.set_xlabel("Subject")
    ax.set_title(f"IC {i + 1}", fontsize=10)
    fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.04, label="loading")
axes[0].set_ylabel("Frequency (Hz)")
fig.suptitle(
    f"Subject × Frequency per IC (channel-avg scores) — {LABEL}",
    fontsize=13,
    y=1.02,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "iva_pairmap_subject_frequency.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")


# ── (3) Subject × Channel — x=subject, y=channel. Frequency-avg scores. ──
# iva_scores.mean(axis=-2) → (S, K, C); transpose to (K, C, S) for plot.
subj_chan = iva_scores.mean(axis=-2).transpose(1, 2, 0)[:n_show]  # (K, C, S)
_vlim = _safe_vlim(subj_chan)

fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 5.0), sharey=True)
if n_show == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    mesh = ax.pcolormesh(
        subjects_idx,
        channels,
        subj_chan[i],
        cmap="RdBu_r",
        vmin=-_vlim,
        vmax=_vlim,
        shading="auto",
    )
    ax.set_xticks(subjects_idx)
    ax.set_xticklabels(subject_labels, fontsize=8)
    ax.set_xlabel("Subject")
    ax.set_title(f"IC {i + 1}", fontsize=10)
    fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.04, label="loading")
axes[0].set_ylabel("Channel")
fig.suptitle(
    f"Subject × Channel per IC (frequency-avg scores) — {LABEL}",
    fontsize=13,
    y=1.02,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "iva_pairmap_subject_channel.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")


# ── (4) Frequency × Channel — x=frequency, y=channel. Matches 04 ICA cell 33. ─
# Subject-averaged scores: (K, F, C); transposed to (C, F) at plot time.
freq_chan = iva_scores.mean(axis=0)[:n_show]  # (K, F, C)

fig, axes = plt.subplots(1, n_show, figsize=(3.5 * n_show, 4.5), sharey=True)
if n_show == 1:
    axes = [axes]
for i, ax in enumerate(axes):
    data_i = freq_chan[i].T  # (C, F)
    vlim_i = _safe_vlim(data_i)
    mesh = ax.pcolormesh(
        FREQS,
        channels,
        data_i,
        cmap="RdBu_r",
        vmin=-vlim_i,
        vmax=vlim_i,
        shading="auto",
    )
    ax.set_xlabel("Frequency (Hz)")
    ax.set_title(f"IC {i + 1}", fontsize=10)
    fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.04, label="loading")
axes[0].set_ylabel("Channel")
fig.suptitle(
    f"Frequency × Channel per IC (subject-avg scores) — {LABEL}",
    fontsize=13,
    y=1.02,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "iva_pairmap_frequency_channel.png",
        dpi=150,
        bbox_inches="tight",
    )
plt.show()
plt.close("all")

---
## Analysis (j) — Mean Subject Loading per Component

For each IVA component the **mean absolute component value over
time** per subject — a scalar per `(subject, component)` pair
summarising how strongly each participant's time profile expresses
the kth source across the recording:

```
subject_loadings[s, k] = mean over t of |iva_components[s, k, t]|
```

Subjects with uniformly high loadings indicate a stimulus-driven
mode; uneven loadings reflect individual differences. Mirrors 04
ICA cell 21, with the source variable swapped from `scores_3d`
(scores in time) to `iva_components` (components in time — same
axis semantics, different IVA role).

| Quantity | Shape | Description |
|----------|-------|-------------|
| `subject_loadings` | `(S, N_PCA)` | Mean `|component|` over time, per subject and component |

In [ ]:
# Per-subject mean |component| over time: (S, K)
subject_loadings = np.abs(iva_components).mean(axis=2)  # (S, K)

n_show = N_COMPONENTS_SHOW
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 4), sharey=True)
if n_show == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.barh(
        range(n_subjects),
        subject_loadings[:, i],
        color="darkorange",
    )
    ax.set_yticks(range(n_subjects))
    ax.set_yticklabels([f"S{s + 1}" for s in range(n_subjects)], fontsize=8)
    ax.set_xlabel("|score|")
    ax.set_title(f"IC {i + 1}", fontsize=10)

axes[0].set_ylabel("Subject")
fig.suptitle(
    f"Per-Subject Mean Loading per Component — {LABEL}",
    fontsize=13,
    y=1.02,
)
fig.tight_layout()
if SAVE_PLOTS:
    fig.savefig(
        PLOTS_DIR / "iva_subject_loadings.png", dpi=150, bbox_inches="tight"
    )
plt.show()
plt.close("all")